## The downward API — pod metadata as env vars or files

Sometimes a container needs to know something *about itself* — its own pod name, namespace, IP, node, labels, or resource limits. None of that is in a ConfigMap or Secret; it's in the pod's own object. The **downward API** exposes it. Two surfaces.

**As env vars — `fieldRef` / `resourceFieldRef`:**

```yaml
env:
  - name: POD_NAME
    valueFrom: { fieldRef: { fieldPath: metadata.name } }
  - name: POD_IP
    valueFrom: { fieldRef: { fieldPath: status.podIP } }
  - name: NODE_NAME
    valueFrom: { fieldRef: { fieldPath: spec.nodeName } }
  - name: CPU_LIMIT
    valueFrom:
      resourceFieldRef: { resource: limits.cpu, divisor: 1m }
```

**As files in a volume — `downwardAPI`:**

```yaml
volumes:
  - name: podinfo
    downwardAPI:
      items:
        - { path: labels, fieldRef: { fieldPath: metadata.labels } }
```

Mount it and the container reads its labels/annotations from files — useful when an app expects "config" but really wants "which environment am I in?"

Available fields: `metadata.name/namespace/uid/labels/annotations`, `spec.nodeName/serviceAccountName`, `status.hostIP/podIP/podIPs`; `resourceFieldRef` gives `limits.cpu/memory`, `requests.*` — handy for sizing a JVM heap against the actual pod limit.

On our map this closes the Config box: alongside ConfigMap and Secret projecting *external* data into a Pod, the downward API projects the Pod's *own* facts into it — same **env / mount** chip, opposite direction.